# Lane Agent Runner

이 노트북은 MMS LAS 차선 탐지 에이전트를 **주피터 노트북에서 바로 실행**하기 위한 파일입니다.

실행 순서:
1. 아래 설치 셀 실행
2. `PROJECT_ROOT`, `LAS_PATH`, `P0`, `P1`, `OUTPUT_CSV` 수정
3. 전체 실행


In [ ]:
# 필요 패키지 설치
# 처음 한 번만 실행하면 됩니다.
# %pip install -q numpy laspy matplotlib


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np

# 프로젝트 루트 경로
# 1) 노트북을 lane_agent_project 폴더 안에 복사해서 열면 Path.cwd() 사용
# 2) 아니면 아래처럼 직접 경로 지정
PROJECT_ROOT = Path.cwd()
# PROJECT_ROOT = Path(r"D:/your_folder/lane_agent_project")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# 실행 파라미터 설정
LAS_PATH = PROJECT_ROOT / 'data' / '5959_202504030857_S02_01.las'   # <- 실제 LAS 경로로 수정
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'
OUTPUT_CSV = PROJECT_ROOT / 'lane_points.csv'

# 시작점 / 방향점 (예시값, 반드시 실제 값으로 바꿔야 함)
P0 = np.array([314463.621002, 3899692.699001, 74.064003], dtype=np.float64)
P1 = np.array([314460.868988, 3899692.244999, 74.109001], dtype=np.float64)

print('LAS_PATH   =', LAS_PATH)
print('CONFIG_PATH=', CONFIG_PATH)
print('OUTPUT_CSV =', OUTPUT_CSV)
print('P0 =', P0)
print('P1 =', P1)


In [ ]:
# 모듈 import
from lane_agent.config import load_config
from lane_agent.las_io import load_las_xyz_intensity
from lane_agent.agent import LaneTrackerAgent
from lane_agent.csv_io import save_xyz_csv


In [ ]:
# 설정 / LAS 로드
cfg = load_config(CONFIG_PATH)
las = load_las_xyz_intensity(str(LAS_PATH))

print('num_points =', las.xyz.shape[0])
print('xyz shape  =', las.xyz.shape)
print('intensity shape =', las.intensity.shape)
print('config =', cfg)


In [ ]:
# 에이전트 실행
agent = LaneTrackerAgent(las.xyz, las.intensity, cfg)
debug_path = OUTPUT_CSV.with_suffix(OUTPUT_CSV.suffix + '.debug.json')

result = agent.track(P0, P1, str(debug_path))
save_xyz_csv(OUTPUT_CSV, result.points)

print('Saved CSV    :', OUTPUT_CSV)
print('Saved debug  :', debug_path)
print('Stop reason  :', result.stop_reason)
print('Num lane pts :', result.points.shape[0])


In [ ]:
# 결과 미리보기
import csv

rows = []
with open(OUTPUT_CSV, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        rows.append(row)
        if i >= 10:
            break

for row in rows:
    print(row)


In [ ]:
# 디버그 정보 일부 보기
if debug_path.exists():
    dbg = json.loads(debug_path.read_text(encoding='utf-8'))
    print('stop_reason =', dbg.get('stop_reason'))
    print('num_points   =', dbg.get('num_points'))
    print('num_steps    =', len(dbg.get('steps', [])))
    if dbg.get('steps'):
        print('first step keys =', dbg['steps'][0].keys())
else:
    print('debug file not found')


In [ ]:
# 간단한 2D 시각화 (XY)
import matplotlib.pyplot as plt

if result.points.shape[0] > 0:
    plt.figure(figsize=(8, 8))
    plt.plot(result.points[:, 0], result.points[:, 1], marker='o', markersize=2)
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title('Tracked Lane Polyline (XY)')
    plt.axis('equal')
    plt.grid(True)
    plt.show()
else:
    print('No tracked points.')
